# E-Commerce Lakehouse: Gold Business Metrics

## Monthly sales for delivered orders

Group delivered orders by purchase month.
Report merchandise sales, freight, recorded payments, and average
merchandise value per order separately.

Amounts are in Brazilian reais (BRL). Sales represent source item
prices for delivered orders, not accounting net revenue or profit.
Incomplete purchase months and undelivered orders affect comparisons.

In [0]:
from pyspark.sql import functions as F

gold_orders_df = spark.table(
    "workspace.ecommerce.gold_order_summary"
)

delivered_orders_df = gold_orders_df.filter(
    F.col("order_status") == "delivered"
)

monthly_sales_df = (
    delivered_orders_df
    .groupBy("purchase_month")
    .agg(
        F.count("*").alias("delivered_order_count"),
        F.countDistinct("customer_unique_id").alias("customer_count"),
        F.sum("merchandise_value").alias("merchandise_sales_brl"),
        F.sum("freight_value").alias("freight_brl"),
        F.sum("recorded_payment_value").alias("recorded_payments_brl"),
        F.round(
            F.avg("merchandise_value"), 2
        ).alias("avg_merchandise_per_order_brl"),
        F.count(
            F.when(F.col("missing_items_flag"), 1)
        ).alias("orders_missing_items"),
        F.count(
            F.when(F.col("missing_payments_flag"), 1)
        ).alias("orders_missing_payments")
    )
)

(
    monthly_sales_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.ecommerce.gold_monthly_sales")
)

# Reconcile the monthly order counts with the delivered population.
monthly_order_count = monthly_sales_df.agg(
    F.sum("delivered_order_count").alias("orders")
).first()["orders"]

delivered_order_count = delivered_orders_df.count()

print("Delivered source orders:", delivered_order_count)
print("Orders represented in monthly metrics:", monthly_order_count)

if monthly_order_count != delivered_order_count:
    raise ValueError("Monthly order counts do not reconcile.")

display(monthly_sales_df.orderBy("purchase_month"))

Delivered source orders: 96478
Orders represented in monthly metrics: 96478


purchase_month,delivered_order_count,customer_count,merchandise_sales_brl,freight_brl,recorded_payments_brl,avg_merchandise_per_order_brl,orders_missing_items,orders_missing_payments
2016-09-01,1,1,134.97,8.49,null,134.97,0,1
2016-10-01,265,262,40325.11,6165.55,46566.71,152.17,0,0
2016-12-01,1,1,10.90,8.72,19.62,10.90,0,0
2017-01-01,750,718,111798.36,15684.01,127545.67,149.06,0,0
2017-02-01,1653,1630,234223.40,37015.92,271298.65,141.70,0,0
2017-03-01,2546,2508,359198.85,55132.10,414369.39,141.08,0,0
2017-04-01,2303,2274,340669.68,50142.72,390952.18,147.92,0,0
2017-05-01,3546,3479,489338.25,77513.15,567066.73,138.00,0,0
2017-06-01,3135,3076,421923.37,68127.00,490225.60,134.58,0,0
2017-07-01,3872,3802,481604.52,84694.56,566403.93,124.38,0,0


## Product category performance

Join delivered order items to Silver products and summarize sales
by the original product category name. Label missing categories
as unknown without excluding their sales.

Count item rows and distinct orders separately. An order containing
multiple categories can contribute to more than one category,
so category order counts must not be added together.

In [0]:
items_df = spark.table("workspace.ecommerce.silver_order_items")
products_df = spark.table("workspace.ecommerce.silver_products")

delivered_items_df = items_df.join(
    delivered_orders_df.select("order_id"),
    on="order_id",
    how="inner"
)

category_items_df = (
    delivered_items_df
    .join(
        products_df.select("product_id", "product_category_name"),
        on="product_id",
        how="left"
    )
    .withColumn(
        "product_category",
        F.when(
            F.col("product_category_name").isNull()
            | (F.trim(F.col("product_category_name")) == ""),
            F.lit("unknown")
        ).otherwise(F.col("product_category_name"))
    )
)

category_sales_df = (
    category_items_df
    .groupBy("product_category")
    .agg(
        F.count("*").alias("item_count"),
        F.countDistinct("order_id").alias("order_count"),
        F.countDistinct("product_id").alias("product_count"),
        F.sum("price").alias("merchandise_sales_brl"),
        F.sum("freight_value").alias("freight_brl")
    )
)

source_totals = delivered_items_df.agg(
    F.count("*").alias("items"),
    F.sum("price").alias("sales")
).first()

category_totals = category_sales_df.agg(
    F.sum("item_count").alias("items"),
    F.sum("merchandise_sales_brl").alias("sales")
).first()

print("Delivered source items:", source_totals["items"])
print("Items represented in categories:", category_totals["items"])
print("Delivered merchandise sales:", source_totals["sales"])
print("Category merchandise sales:", category_totals["sales"])

if (
    source_totals["items"] != category_totals["items"]
    or source_totals["sales"] != category_totals["sales"]
):
    raise ValueError("Category totals do not reconcile.")

(
    category_sales_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.ecommerce.gold_category_sales")
)

display(
    category_sales_df.orderBy(
        F.col("merchandise_sales_brl").desc()
    ).limit(10)
)

Delivered source items: 110197
Items represented in categories: 110197
Delivered merchandise sales: 13221498.11
Category merchandise sales: 13221498.11


product_category,item_count,order_count,product_count,merchandise_sales_brl,freight_brl
beleza_saude,9465,8647,2397,1233131.72,178957.81
relogios_presentes,5859,5495,1300,1166176.98,98156.14
cama_mesa_banho,10953,9272,2991,1023434.76,201774.50
esporte_lazer,8431,7530,2822,954852.55,163404.36
informatica_acessorios,7644,6530,1600,888724.61,143999.16
moveis_decoracao,8160,6307,2593,711927.69,168402.23
utilidades_domesticas,6795,5743,2282,615628.69,142763.56
cool_stuff,3718,3559,770,610204.10,81476.79
automotivo,4140,3810,1853,578966.65,90488.10
brinquedos,4030,3804,1373,471286.48,75774.58


## Customer purchase metrics

Summarize delivered orders by customer_unique_id to recognize
customers across purchases.

Calculate delivered order count, merchandise sales, and first
and latest purchase dates. Repeat customers have at least two
delivered orders in this dataset's observation period.

In [0]:
customer_metrics_df = (
    delivered_orders_df
    .groupBy("customer_unique_id")
    .agg(
        F.count("*").alias("delivered_order_count"),
        F.sum("merchandise_value").alias("merchandise_sales_brl"),
        F.min("purchase_date").alias("first_purchase_date"),
        F.max("purchase_date").alias("latest_purchase_date")
    )
    .withColumn(
        "repeat_customer_flag",
        F.col("delivered_order_count") >= 2
    )
)

customer_totals = customer_metrics_df.agg(
    F.count("*").alias("customers"),
    F.sum("delivered_order_count").alias("orders"),
    F.sum("merchandise_sales_brl").alias("sales"),
    F.count(
        F.when(F.col("repeat_customer_flag"), 1)
    ).alias("repeat_customers")
).first()

source_totals = delivered_orders_df.agg(
    F.count("*").alias("orders"),
    F.sum("merchandise_value").alias("sales")
).first()

print("Customers with delivered orders:", customer_totals["customers"])
print("Repeat customers:", customer_totals["repeat_customers"])
print("Delivered source orders:", source_totals["orders"])
print("Orders represented in customer metrics:", customer_totals["orders"])
print("Source merchandise sales:", source_totals["sales"])
print("Customer merchandise sales:", customer_totals["sales"])

if (
    source_totals["orders"] != customer_totals["orders"]
    or source_totals["sales"] != customer_totals["sales"]
):
    raise ValueError("Customer metrics do not reconcile.")

(
    customer_metrics_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.ecommerce.gold_customer_metrics")
)

display(
    customer_metrics_df.orderBy(
        F.col("merchandise_sales_brl").desc()
    ).limit(10)
)

Customers with delivered orders: 93358
Repeat customers: 2801
Delivered source orders: 96478
Orders represented in customer metrics: 96478
Source merchandise sales: 13221498.11
Customer merchandise sales: 13221498.11


customer_unique_id,delivered_order_count,merchandise_sales_brl,first_purchase_date,latest_purchase_date,repeat_customer_flag
0a0a92112bd4c708ca5fde585afaa872,1,13440.00,2017-09-29,2017-09-29,false
da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,2017-04-01,2017-04-01,true
763c8b1c9c68a0229c42c9fc6f662b93,1,7160.00,2018-07-15,2018-07-15,false
dc4802a71eae9be1dd28f5d788ceb526,1,6735.00,2017-02-12,2017-02-12,false
459bef486812aa25204be022145caa62,1,6729.00,2018-07-25,2018-07-25,false
ff4159b92c40ebe40454e3e6a7c35ed6,1,6499.00,2017-05-24,2017-05-24,false
4007669dec559734d6f53e029e360987,1,5934.60,2017-11-24,2017-11-24,false
eebb5dda148d3893cdaf5b5ca3040ccb,1,4690.00,2017-04-18,2017-04-18,false
48e1ac109decbb87765a3eade6854098,1,4590.00,2018-06-22,2018-06-22,false
a229eba70ec1c2abef51f04987deb7a5,1,4400.00,2018-05-31,2018-05-31,false


## Delivery performance by customer state

Summarize delivered orders by customer state. Report average
calendar days from purchase to delivery and carrier transit time.

Calculate late delivery percentage only among orders with known
actual and estimated delivery dates. Exclude flagged shipping
sequences from carrier transit averages, while retaining those
orders in other eligible metrics.

In [0]:
delivery_metrics_df = (
    delivered_orders_df
    .groupBy("customer_state")
    .agg(
        F.count("*").alias("delivered_order_count"),
        F.count("delivery_days").alias("orders_with_delivery_duration"),
        F.round(
            F.avg("delivery_days"), 2
        ).alias("avg_delivery_days"),
        F.count("carrier_transit_days").alias("orders_with_valid_transit"),
        F.round(
            F.avg("carrier_transit_days"), 2
        ).alias("avg_carrier_transit_days"),
        F.count("late_delivery_flag").alias("orders_with_lateness_check"),
        F.count(
            F.when(F.col("late_delivery_flag"), 1)
        ).alias("late_order_count"),
        F.count(
            F.when(F.col("shipping_sequence_issue"), 1)
        ).alias("shipping_sequence_issue_count")
    )
    .withColumn(
        "late_delivery_pct",
        F.when(
            F.col("orders_with_lateness_check") > 0,
            F.round(
                100.0 * F.col("late_order_count")
                / F.col("orders_with_lateness_check"),
                2
            )
        )
    )
)

summary = delivery_metrics_df.agg(
    F.sum("delivered_order_count").alias("orders"),
    F.sum("orders_with_lateness_check").alias("eligible_orders"),
    F.sum("late_order_count").alias("late_orders"),
    F.sum("shipping_sequence_issue_count").alias("shipping_issues")
).first()

source_order_count = delivered_orders_df.count()

print("Delivered source orders:", source_order_count)
print("Orders represented in delivery metrics:", summary["orders"])
print("Orders eligible for lateness checks:", summary["eligible_orders"])
print("Late orders:", summary["late_orders"])
print("Shipping sequence issues:", summary["shipping_issues"])

if summary["orders"] != source_order_count:
    raise ValueError("Delivery order counts do not reconcile.")

(
    delivery_metrics_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.ecommerce.gold_delivery_metrics")
)

display(
    delivery_metrics_df.orderBy(
        F.col("delivered_order_count").desc()
    )
)

Delivered source orders: 96478
Orders represented in delivery metrics: 96478
Orders eligible for lateness checks: 96470
Late orders: 6534
Shipping sequence issues: 23


customer_state,delivered_order_count,orders_with_delivery_duration,avg_delivery_days,orders_with_valid_transit,avg_carrier_transit_days,orders_with_lateness_check,late_order_count,shipping_sequence_issue_count,late_delivery_pct
SP,40501,40494,8.7,40479,5.56,40494,1820,14,4.49
RJ,12350,12350,15.24,12349,11.95,12350,1495,1,12.11
MG,11354,11354,11.94,11349,8.72,11354,519,5,4.57
RS,5345,5344,15.25,5343,12.03,5344,325,1,6.08
PR,4923,4923,11.94,4923,8.73,4923,199,0,4.04
SC,3546,3546,14.9,3544,11.58,3546,291,2,8.21
BA,3256,3256,19.28,3256,15.99,3256,396,0,12.16
DF,2080,2080,12.9,2080,9.74,2080,118,0,5.67
ES,1995,1995,15.72,1995,12.35,1995,214,0,10.73
GO,1957,1957,15.54,1957,12.38,1957,128,0,6.54
